# Package a model-facing feature pipeline

Build the pure, inspectable layer that sits in front of a model: feature projection, frozen preprocessing, validated training options and artifact compatibility checks. No model is trained here. Use the same feature code in training and serving.

Fill in each starter, then run its check cell. Restart the kernel and run all cells before submitting. Keep your interpretation in the last Markdown cell.


## 1. Project model features

Project fields in schema order.

Implement feature_vector(row, columns). Return values in columns order. Ignore extra fields. Raise KeyError for required fields that are absent.


In [ ]:
def feature_vector(row, columns):
    raise NotImplementedError


In [ ]:
assert feature_vector({"b":2,"a":1,"c":5},["a","b"]) == [1,2]
assert feature_vector({"a":None},["a"]) == [None]
assert feature_vector({},[]) == []
assert feature_vector({"a":1,"b":2},["b","a"]) == [2,1], "Follow the order of columns, not the order of the row"
try:
    feature_vector({},["missing"])
except KeyError:
    pass
else:
    raise AssertionError("Require declared fields")
print("OK")


## 2. Apply frozen scaler state

Apply a frozen transform.

Implement apply_scaler(values, centers, scales). Require matching lengths and positive finite scales. Inputs are finite. Return each centred value divided by the fitted scale.


In [ ]:
def apply_scaler(values, centers, scales):
    raise NotImplementedError


In [ ]:
assert apply_scaler([9,5],[5,5],[2,1]) == [2,0]
assert apply_scaler([],[],[]) == []
for args in [([1],[0,1],[1]),([1],[0],[0]),([1],[0],[float("inf")]),([1],[0],[float("nan")]),([1],[1],[1,2])]:
    try:
        apply_scaler(*args)
    except ValueError:
        pass
    else:
        raise AssertionError("Reject invalid state")
print("OK")


## 3. Resolve a training configuration

Validate explicit training options.

Implement training_config(options). Recognise only epochs and rate, with defaults 10 and 0.01. Epochs must be a positive exact integer; rate a number, excluding bool, in (0,1]. Return a fresh dictionary and normalise rate to float.


In [ ]:
def training_config(options):
    raise NotImplementedError


In [ ]:
assert training_config({}) == {"epochs":10,"rate":0.01}
assert training_config({"epochs":1,"rate":1}) == {"epochs":1,"rate":1.0}
for bad in ({"epoch":2},{"epochs":True},{"rate":"0.01"},{"rate":False},{"rate":0},{"epochs":2.5},{"epochs":0},{"rate":1.5}):
    try:
        training_config(bad)
    except ValueError:
        pass
    else:
        raise AssertionError("Reject invalid training options")
o = {"epochs":2}
r = training_config(o)
assert r is not o and o == {"epochs":2}, "Return a fresh dictionary and leave the input unchanged"
print("OK")


## 4. Check artifact compatibility

Check an artifact against its consumer.

Implement compatible_artifact(manifest, schema, features). Return true only when the schema version equals the expected version and the stored feature list equals the expected ordered list. Extra fields are allowed.


In [ ]:
def compatible_artifact(manifest, schema, features):
    raise NotImplementedError


In [ ]:
assert compatible_artifact({"schema":1,"features":["a","b"],"version":"v1"},1,["a","b"])
assert not compatible_artifact({"schema":1,"features":["b","a"]},1,["a","b"])
assert not compatible_artifact({"schema":2,"features":["a"]},1,["a"])
assert not compatible_artifact({},1,[])
print("OK")


## 5. Transform a serving row from a compatible fitted artifact

Implement serving_features(row, manifest, schema, columns). Raise ValueError for incompatible
metadata before attempting feature access. For a compatible artifact, project columns in order and
apply the frozen centers and scales stored in the manifest.


In [ ]:
def serving_features(row, manifest, schema, columns):
    raise NotImplementedError


In [ ]:
manifest = {"schema":1,"features":["b","a"],"centers":[2,5],"scales":[2,1]}
assert serving_features({"a":5,"b":6},manifest,1,["b","a"]) == [2,0]
try:
    serving_features({},manifest,1,["a","b"])
except ValueError:
    pass
else:
    raise AssertionError("Check artifact compatibility before projection")
try:
    serving_features({"a":5,"b":6},{**manifest,"schema":2},1,["b","a"])
except ValueError:
    pass
else:
    raise AssertionError("Reject a manifest with the wrong schema version")
print("OK")


## Interpretation

Sketch the modules and call boundaries. Explain where learned state lives and how an older incompatible artifact would be handled.

**Your answer**
